## Convolutional Layers

In [45]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images
import torchvision
import torchvision.transforms.v2 as T
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import fetch_openml
from functools import partial

load images, images are represented as a plain python list of numpy arrays (2 seperate arrays), so stack them into a single numpy array (with another dimension) and covnert it to a tensor and rescale

In [14]:
sample_images = np.stack(load_sample_images()['images'])
print(type(sample_images), sample_images.shape)
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255
print(type(sample_images))

<class 'numpy.ndarray'> (2, 427, 640, 3)
<class 'torch.Tensor'>


2 images, 427x640 pixles, w/ rgb color channels. pytorch expects channel dim to be before the h/w dims, so need to permute it

also use torch vision `CenterCrop` to center-crop the images, to get the 70x120 pixels in the center

In [16]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [19]:
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

now let's create a 2d convolutional layer and feed it these images
- 32 filters, each of size 7x7 (kernel_size=7)

all 32 filters slide over the image (all 3 channels included), at each position the the filter overlaps a 7x7 patch of the iamge and computes a single number. that num becomes 1 pixle in the filters output feature map. then slides over by 1 pixel (default stride=1), till it covers every postion in the image, and then full feature map is created. 

this happens 32 times, so we get 32 differnt feature maps stacked together, each looking for something different.

In [23]:
torch.manual_seed(42)

conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, padding='same')
fmaps = conv_layer(cropped_images)

In [24]:
fmaps.shape

torch.Size([2, 32, 70, 120])

In [ ]:
conv_layer.weight.shape # 1 weight per pixel in 7x7 grid, 3 layers deep [output channels, input channels, kernel h, kernel w]
# note that all the neurons in the output feature maps share the same weights, each 7x7 kernel looks for 1 specific thing
# so it can be reused anywhere in the image

torch.Size([32, 3, 7, 7])

In [ ]:
conv_layer.bias.shape # 1 bias for each output channel (set of weights) [output channels]

torch.Size([32])

also we need to add an activation function after each conv layer, since each layer performs a linear operation so they can combine tg and its js 1 layer if we had no activation function. activation function helps it learn complex patterns. now we look at pooling layers

## Pooling Layers

goal of pooling layer is to *subsample*, that is shrink the input image to reduce computational load, mem usage, and # of params (limit risk of overfitting)

each neuron in a pooling layer connected to outputs of a limit # of neurons from prev layer, w/ small rectangular receptive field. but pooling neuron has no weights or biases, it just *aggregates* the inputs (i.e. combines them together) using a function like max or mean. It also introduces some *invariance*, e.g. small translations won't affect the outputs of the CNN if we don't want them to

downsides are that it's destructive, drops input values, and sometimes we don't want invariance, we want to detect small changes

also pooling can be done along depth, e.g. learning from diff filters that detected a diagonal line at diff rotations. the depth pooling then js says at this spot here did any of the filters detect a diagonal line. yes? ok so theres a digaonal line here. introduces invariance.

now let's implement some pooling layers

we will create a pooling layer using a Max aggregate function and 2x2 kernel, w/ the stride defaulting to kernel size (2), and default padding=0. 

This padding layer will take our inputs, and return the max value of a 2x2 grid, therefore only taking 1 out of every 4 pixels from the input values

Max is good b/c it takes the strongest signal from a grid, getting rid of ones that are less confident and would prob convey less info and not be as helpful to us

In [27]:
max_pool = nn.MaxPool2d(kernel_size=2)

lets also make a depthwise pooling layer. we need to make a custom module for this.

In [40]:
class DepthPool(torch.nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size # default to ker size if custom not provided
        self.padding = padding # default 0

    def forward(self, inputs):
        batch, channels, height, width = inputs.shape
        Z = inputs.reshape(batch, channels, height * width) # merge spatial dims, so we have all our pixels in 1d
        Z = Z.permute(0, 2, 1) # switch spatial and channels dims so channel is last dim
        Z = F.max_pool1d(Z, kernel_size=self.kernel_size, stride=self.stride, 
                         padding=self.padding) # compute max pool along last dimension (1d) which is CHANNEL for each pixel
        Z = Z.permute(0, 2, 1) # switch back spatial and channels dims
        return Z.reshape(batch, -1, height, width) # unmerge spatial dims, back into 2d image

In [43]:
dp = DepthPool(kernel_size=4)
test = dp(fmaps)
test.shape

torch.Size([2, 8, 70, 120])

another type of pooling layer is the global avg pooling layer, which takes the avg of all the input values and returns just 1. good before output layer, bad o/w cuz it throws away all the info.  use `nn.AdaptiveAvgPool2d` which lets us specific desired spatial dims of output and adapts kernel size to get desired result. or we could use `torch.mean()`, get avg of the h and w dims

## CNN Architectures

Typically CNN's stack a few conv layers (generally followed by ReLU), then pooling, then more conv, then pooling, etc etc. 

Image get's smaller as it passes thru, but also gets deeper (more feature maps), so layer in the network it's more descriptive than just raw pictures. e.g. over here theres a diagonal line, that's also a boundry between light and dark, etc. 

At the top, a regular feedforward neural network is added w/ a few fully connected layers and the final layer ouputs the prediction. e.g. a softmax layer that outputs class probabilities for mutliclass classification

An example of a basic CNN is this one for the Fashion MNIST dataset. 

In [ ]:
# our default 2d conv layer, use partial to create a new func of a thats a Conv2d layer w/ some prefill args so we dont repeat
#   we can fill in needed ones as neccesary
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding='same') 
model  = nn.Sequential( # sequential model like in our mlps to chain layers tg
    # conv and pooling layers
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2), # pool, take max pixel from every 2
    DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(), # more conv and pooling follow
    DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
    # output channels grow, simple features can be combined to make many more complex ones
    DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(), 
    nn.MaxPool2d(kernel_size=2), # image is 3x3 now from 28x28
    nn.Flatten(), # flatten it for the linear, linear can't understand 2d
    # image is 3x3 pixels, 256 feature maps per pixel (3x3x256 = 2304) is size of input for linear
    # now fully connected network 2 dense hidden layers, 10 output neurons (10 class)
    nn.Linear(in_features=2304, out_features=128), nn.ReLU(), 
    nn.Dropout(0.5), # dropout layer to reduce overfitting 
    nn.Linear(in_features=128, out_features=64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10), # leave out softmax so output logits, use xentropy as loss 
).to('mps')


skip train and evaluating it, done before a bunch, want to focus on CNN architecture. achieves 92% acc, better than a normal dense network.

There's lot's of diff CNN architectures we can choose from, this is just a super basic one. 

## Implement ResNet-34 CNN

let's implement a ResNet-34 from scratch. first we will create a `ResidualUnit` layer. 

normal deep networks derive the whole target function, meaning that all the layers in something deep have to perfectly mesh together, and if it get's too deep it can mess up and loose info b/c of so many operations happening. 

they also force every layer to rederive don't do anything from scratch if notIs a good explanationhing is needed at that point, which is hard since the matmul, bias, relu has to perfectly cancel over many layers, and if it does something we dont want it can lose the meaning of the data. 

resnet gives each layer the original input and asks it to learn optional tweaks to stack on top aka the residual, which is ez, instaed of transforming the entire input. 

output = (small learning adjustment) + (original input)

also, this means at the start the output is just the identity function (since the weights at the start are close to zero, so the model calculates the residual as ~0 -> f(x) + x = ~x).
so if the target function is close to the identity function, the model just has to learn the residual (difference between target and identity) rather than the entire target function, which is typically easier. so the model learns f(x) = h(x) - x, and the output is f(x) + x = h(x) where h(x) is the target function. 
- from this, learning "do nothing" for a group of layers in a very deep network is much easier, since their residual is just 0. 
- so their learn f(x) = 0, and output f(x) + x = 0 + x = x, so it really did not do anything. 

The `ResidualUnit` layer is the thing that packages this whole mechanism up.

In [46]:
class ResidualUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        DefaultConv2d = partial( # default arg conv layer function
            nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False # bias false cuz batchnorms makes reduant & it has own shift
        )
        # f(x), the learned adjustment / residual 
        self.main_layers = nn.Sequential( # the main stack of layers
            DefaultConv2d(in_channels=in_channels, out_channels=out_channels, stride=stride),
            # batchnorm is a layer that normalizes its input dist, keeping it consistent throughout all the layers
            # good for us cuz w/ many layers the distribution of output values can shift
            # this keeps it all the same over all our batches over all the layers
            # -> trains faster and more reliably
            # -> note has adjustable params so we can adjust the dist but keep it consistent
            nn.BatchNorm2d(out_channels), 
            nn.ReLU(),
            DefaultConv2d(in_channels=out_channels, out_channels=out_channels),
            nn.BatchNorm2d(out_channels),
        )
        # shortcut path (2 cases)
        # channels/spatial size changes so cant add raw x to next layer (size wont match up) 
        # projection shortcut, 1x1 conv w/ same stride to reshape input to new shape 
        # 1x1 kernel so only looks at 1 input pixel at a time, combining channels lienarly
        # no relu, 2nd layer, js batchnorm to normalize. don't wanna do any pattern learning here other than the min to reshape
        # we r trying not to learn, just wanna reshape. so output is as close as input as possible js reshaped 
        if stride > 1: 
            self.skip_connection = nn.Sequential( # skip layers
                DefaultConv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1,
                              stride=stride, padding=0),
                nn.BatchNorm2d(out_channels),
            )
        else: # idenity module (returns inputs unchanged) if stride is 1, shapes alr match no need to change
            self.skip_connection = nn.Identity()

    # actual f(x) + x, conv stack gives us f(x), the residual, and shortcut passes x through. 
    # combine and relu for h(x), our target
    def forward(self, inputs):
        return F.relu(self.main_layers(inputs) + self.skip_connection(inputs))

Now we have our residual unit layer for the main learning stack for the residual and the shortcut/skip connection for x
we can build out the actual ResNet34 CNN architecture module

In [47]:
class ResNet34(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [ # add our first few layers at the bottom (conv and pool)
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2,
                      padding=3, bias=False),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        ]
        # now we need to add our RU layers w/ varying amts of filters
        # if the # of filters is the same as the prev RU the stride = 1, 2 o/w 
        prev_filters = 64 # the # of filters the prev had for the first block of RUs
        # increase the # of filters per RU further down the network
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3: # corresponding amt of RU's for each filter amt
            stride = 1 if filters == prev_filters else 2 # if new filter size, increase stride (shrink spatial size), same o/w
            layers.append(ResidualUnit(prev_filters, filters, stride=stride)) # add the RU
            prev_filters = filters
        layers += [ 
            nn.AdaptiveAvgPool2d(output_size=1), # take global avg pool of whatever left of image
            nn.Flatten(), # flatten all filters and avg to vector
            nn.LazyLinear(10), # 1 linear fully connected layer w/ 10 outputs, no final activation so outputs logits
        ]
        self.resnet = nn.Sequential(*layers) # unpack all layers into a sequential module to apply sequentially

    def forward(self, inputs):
        return self.resnet(inputs) # call our sequential module w/ all our layers on the inputs


## TorchVision's Pretrained Models

can also load pretrained models via torchvision. for ex load this ConvNeXt model pretrained on ImageNet 
and use it to classifiy our two sample images. but we need to ensure they r preprocessed how the model expects (right size, 224x224)

In [48]:
device = 'mps'
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_base(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /Users/zainbabar/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100.0%


In [49]:
transforms = weights.transforms() # use transforms designed for this model
preproccessed_images = transforms(sample_images_permuted)

In [50]:
model.eval() # move to eval mode before passing
with torch.no_grad(): # disable autograd, so doesn't track computation graph
    y_logits = model(preproccessed_images.to(device)) # move imgaes to gpu before passing to model

reuslt is a 2x1000 tensor w/ class logits for each image (2) (ImageNet has 1k classes, so gives us score for all 1k classes)

In [53]:
print(y_logits.shape)
y_logits

torch.Size([2, 1000])


tensor([[ 0.9411, -0.3203, -0.3067,  ...,  0.2111,  0.4626, -0.2570],
        [-0.6901, -0.6672, -0.8607,  ...,  0.9288,  4.1106, -0.8371]],
       device='mps:0')

use torch.argmax to get predicted class id (as a number), i.e. the class w/ the max logit. `dim=1` so we choose on columns (the classes)

get class names from the weights object to see what it is

In [54]:
y_pred = torch.argmax(y_logits, dim=1)
y_pred

tensor([698, 985], device='mps:0')

In [ ]:
class_names = weights.meta['categories'] # meta attribute -> metadata dictionary w/ (class_id, class_name)
[class_names[class_id] for class_id in y_pred] # pretty close

['palace', 'daisy']

also look at top3 preds w/ topk() and their estimated probs via softmax (not overall, just between these 3)

In [56]:
y_top3_logits, y_top3_class_ids = y_logits.topk(k=3, dim=1)
[[class_names[class_id] for class_id in top3] for top3 in y_top3_class_ids] # layered dict and list comprehension

[['palace', 'monastery', 'lakeside'], ['daisy', 'pot', 'ant']]

In [57]:
y_top3_logits.softmax(dim=1)

tensor([[0.8618, 0.1185, 0.0197],
        [0.8106, 0.0964, 0.0930]], device='mps:0')

## Pretrained Models for Transfer Learning

we can reuse lower layers of a pretrained model and then replace its classification head with ours and fine tune it for our use. so the common stuff like shapes, basic objects etc we steal, and then just what we need we can add on top.

we will reuse the ConvNeXt model and fine tune it on Flower-s102. 

First let's download dataset via torchvision

In [58]:
DefaultFlowers102 = partial(torchvision.datasets.Flowers102, root="datasets",
                            transform=weights.transforms(), download=True)
train_set = DefaultFlowers102(split="train")
valid_set = DefaultFlowers102(split="val")
test_set = DefaultFlowers102(split="test")

100.0%
100.0%
100.0%


use partial() to avoid repeating args, and transform=weights.transforms() to preprocess images when loaded 

In [59]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

In [ ]:
# class_names = ['pink primrose', ..., 'trumpet creeper', 'blackberry lily']

adapt our pretrained model to this dataset. model was designed to output 1000 logits (pretrained on imagenet) but we only have 102 classes so we need to swap this for a smaller one. 

to find the head (upper layes) use named_children() to find the name of its submodules

In [62]:
[name for name, child in model.named_children()]

['features', 'avgpool', 'classifier']

features is main part, all layers except for global avg pooling layer and the classifier (the head, what we want)

let's take a look at the head

In [63]:
model.classifier

Sequential(
  (0): LayerNorm2d((1024,), eps=1e-06, elementwise_affine=True, bias=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=1024, out_features=1000, bias=True)
)

layer normalization, flatten than a linear w/ 1000 outputs. we wanna swap the linear w/ one that works for us (102 outputs)

In [ ]:
n_classes = 102
model.classifier[2] = nn.Linear(1024, n_classes).to(device)

now retrain it. freeze weights of pretrained layers, don't need too much work with that. train the weights of the head. can fine tune a lil later if neccesary but we wanna do big work on jsut the head.

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

can train the head now by writing a train and eval function. get abt 90% acc w js this. after unfreeze whole model, lower learning rate (typically by a factor of 10) and keep training. we can see what acc we can get w/ this. 

can also perform some data augmentation to increase the amt of training images and make the model better. 